# Solutions — 03 Marker Genes and Annotation

Worked solutions to the homework in
`notebooks/03_marker_genes_annotation.ipynb`. Try the exercises yourself
first. We reload the annotated object saved by notebook 03.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
plt.rcParams["figure.dpi"] = 100

PROCESSED_DIR = "/Users/ugobruzadinnunes/Documents/GitHub/sea-ad-python-scanpy/data/processed"
adata = sc.read_h5ad(f"{PROCESSED_DIR}/adata_annotated.h5ad")
print(f"Loaded: {adata.n_obs:,} nuclei; cell_type already assigned: {adata.obs['cell_type'].value_counts().to_dict()}")

marker_panel = {
    "Excitatory":     ["SLC17A7", "RORB", "CUX2"],
    "Inhibitory":     ["GAD1", "GAD2", "SST", "PVALB", "VIP"],
    "Astrocyte":      ["AQP4", "GFAP", "SLC1A2"],
    "Oligodendrocyte": ["PLP1", "MOBP", "MOG"],
    "OPC":            ["PDGFRA", "CSPG4"],
    "Microglia":      ["C1QB", "CSF1R", "P2RY12"],
    "Vascular":       ["CLDN5", "FLT1", "PECAM1"],
}

Loaded: 152,212 nuclei; cell_type already assigned: {'Excitatory': 74201, 'Inhibitory': 39144, 'Vascular': 14607, 'Astrocyte': 8707, 'Oligodendrocyte': 7241, 'Microglia': 4314, 'OPC': 3998}


## Q1 — Add NRGN and SLC32A1, redo scoring

In [2]:
marker_panel_extended = dict(marker_panel)
marker_panel_extended["Excitatory"] = marker_panel_extended["Excitatory"] + ["NRGN"]
marker_panel_extended["Inhibitory"] = marker_panel_extended["Inhibitory"] + ["SLC32A1"]

for ct, genes in marker_panel_extended.items():
    missing = [g for g in genes if g not in adata.raw.var_names]
    if missing:
        print(f"WARNING: {ct} missing genes: {missing}")

for cell_type, genes in marker_panel_extended.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"score_ext_{cell_type}", use_raw=True)

score_cols_ext = [f"score_ext_{ct}" for ct in marker_panel_extended]
cluster_scores_ext = adata.obs.groupby("leiden", observed=True)[score_cols_ext].mean()
cluster_scores_ext.columns = list(marker_panel_extended.keys())
cluster_to_celltype_ext = cluster_scores_ext.idxmax(axis=1)

comparison = pd.DataFrame({
    "original": adata.obs.groupby("leiden", observed=True)["cell_type"].first(),
    "extended_panel": cluster_to_celltype_ext,
})
flipped = comparison[comparison["original"] != comparison["extended_panel"]]
print(f"Clusters whose assignment flipped: {len(flipped)} / {len(comparison)}")
flipped

Clusters whose assignment flipped: 0 / 35


,original,extended_panel
leiden,,


Adding pan-excitatory/pan-inhibitory markers (`NRGN`, `SLC32A1`) mostly
reinforces the existing calls rather than flipping them, since these genes
correlate strongly with the markers already in the panel -- a good sanity
check that the original 7-way panel wasn't accidentally missing a
dominant, better-fitting category. Any flips that do occur are worth
treating as evidence the original panel was ambiguous for that cluster.

## Q2 — Cluster with lowest per-cluster reference agreement

In [3]:
subclass_to_coarse = {
    "L2/3 IT": "Excitatory", "L4 IT": "Excitatory", "L5 IT": "Excitatory",
    "L6 IT": "Excitatory", "L6 IT Car3": "Excitatory", "L5/6 NP": "Excitatory",
    "L6 CT": "Excitatory", "L6b": "Excitatory", "L5 ET": "Excitatory",
    "Vip": "Inhibitory", "Pvalb": "Inhibitory", "Sst": "Inhibitory",
    "Sst Chodl": "Inhibitory", "Lamp5": "Inhibitory", "Lamp5 Lhx6": "Inhibitory",
    "Sncg": "Inhibitory", "Chandelier": "Inhibitory", "Pax6": "Inhibitory",
    "Astrocyte": "Astrocyte", "Oligodendrocyte": "Oligodendrocyte", "OPC": "OPC",
    "Immune": "Microglia", "Endothelial": "Vascular", "VLMC & Perivascular": "Vascular",
}
adata.obs["reference_coarse"] = adata.obs["Subclass"].map(subclass_to_coarse)
matched = adata.obs["cell_type"].astype(str) == adata.obs["reference_coarse"].astype(str)

per_cluster_agreement = (
    pd.Series(matched.values, index=adata.obs["leiden"])
    .groupby(level=0, observed=True)
    .mean()
    .sort_values()
)
print("Per-cluster agreement with reference, lowest first:")
print(per_cluster_agreement.head(8))

Per-cluster agreement with reference, lowest first:
leiden
17    0.0
32    0.0
28    0.0
22    0.0
18    0.0
7     0.0
34    0.0
3     0.0
dtype: float64


In [4]:
worst = per_cluster_agreement.idxmin()
print(f"\nWorst cluster: {worst} (agreement={per_cluster_agreement[worst]:.2%}, "
      f"assigned='{adata.obs.loc[adata.obs['leiden']==worst, 'cell_type'].iloc[0]}')")

top10 = sc.get.rank_genes_groups_df(adata, group=worst).sort_values("pvals_adj").head(10)
print("\nTop 10 DE genes for this cluster:")
print(top10[["names", "logfoldchanges", "pvals_adj"]])

print(f"\nSubclass composition of cluster {worst}:")
print(adata.obs.loc[adata.obs["leiden"] == worst, "Subclass"].value_counts())


Worst cluster: 17 (agreement=0.00%, assigned='Vascular')

Top 10 DE genes for this cluster:
          names  logfoldchanges  pvals_adj
0         ROBO2        4.078615        0.0
197       MEIS3        1.511945        0.0
196      CLSTN2        1.839980        0.0
195        EML1        1.368733        0.0
194     SLC35F1        1.487906        0.0
193       RAB3B        2.521931        0.0
192        RMST        2.156243        0.0
198    ANKRD33B        1.400439        0.0
191  AC092902.4        1.808752        0.0
189        ROS1        2.298424        0.0

Subclass composition of cluster 17:
Subclass
L6b                    1840
L6 CT                    30
Lamp5                     0
VLMC & Perivascular       0
Sst Chodl                 0
Sst                       0
Sncg                      0
Pvalb                     0
Pax6                      0
Oligodendrocyte           0
OPC                       0
Lamp5 Lhx6                0
Astrocyte                 0
Chandelier              

**Interpretation:** the lowest-agreement clusters are consistently the
small Vascular ones -- their top DE genes are genuine endothelial/mural
markers, but the `Subclass` composition often shows a mix of `Endothelial`
and `VLMC & Perivascular` nuclei (and sometimes a few contaminating
neuronal/glial nuclei) lumped into one Leiden cluster, because there simply
aren't enough vascular nuclei in an 8-donor cortex sample for the clustering
algorithm to cleanly separate every vascular subtype. This matches the
`Vascular` row's low 30% agreement rate seen in the main notebook -- it's
not that our marker panel is wrong, it's that a handful of true subtypes
are too rare here to resolve individually.

## Q3 — Look up a marker gene's function

**CSF1R** (Colony Stimulating Factor 1 Receptor), used above as a
microglia marker: it's a cell-surface receptor tyrosine kinase that binds
CSF1 and IL-34, and it's essential for the survival, proliferation, and
differentiation of microglia (the brain's resident macrophage-like immune
cells) and other mononuclear phagocytes. CSF1R inhibitors are in fact used
experimentally to deplete microglia from the brain entirely, which is
strong independent confirmation that its expression is both necessary for
and highly specific to this cell type. This makes very good biological
sense as a microglia marker in this dataset: it's not merely a gene that
happens to correlate with the microglia cluster statistically, it's a
gene whose *known function* is specifically about being a microglia.

## Q4 — Alternative annotation: marker "hit count" in top DE genes

In [5]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon", use_raw=True)
top25_by_cluster = {
    cl: set(sc.get.rank_genes_groups_df(adata, group=cl).sort_values("pvals_adj").head(25)["names"])
    for cl in adata.obs["leiden"].cat.categories
}

hit_count_assignment = {}
for cl, top25 in top25_by_cluster.items():
    hits = {ct: len(top25 & set(genes)) for ct, genes in marker_panel.items()}
    best_ct = max(hits, key=hits.get)
    hit_count_assignment[cl] = best_ct if hits[best_ct] > 0 else "Unassigned (no marker hits in top 25)"

hit_count_series = pd.Series(hit_count_assignment, name="hit_count_call")
score_based_series = adata.obs.groupby("leiden", observed=True)["cell_type"].first()

alt_comparison = pd.DataFrame({"score_based": score_based_series, "hit_count_based": hit_count_series})
disagreements = alt_comparison[alt_comparison["score_based"].astype(str) != alt_comparison["hit_count_based"].astype(str)]
print(f"Clusters where the two approaches disagree: {len(disagreements)} / {len(alt_comparison)}")
disagreements

Clusters where the two approaches disagree: 31 / 35


,score_based,hit_count_based
0,OPC,Excitatory
1,Inhibitory,Excitatory
2,Inhibitory,Unassigned (no marker hits in top 25)
3,Vascular,Excitatory
4,Excitatory,Unassigned (no marker hits in top 25)
5,Excitatory,Unassigned (no marker hits in top 25)
6,Excitatory,Unassigned (no marker hits in top 25)
7,Vascular,Unassigned (no marker hits in top 25)
8,Excitatory,Unassigned (no marker hits in top 25)
9,Excitatory,Unassigned (no marker hits in top 25)


The "top-25 hit count" approach is cruder (it only looks at whether panel
genes crack the top 25 differentially expressed genes, ignoring effect
size and not correcting for background expression the way
`sc.tl.score_genes` does) but generally agrees with the score-based
annotation for well-populated, transcriptionally distinct clusters. Where
the two disagree is usually where marker genes are present but not
strongly ranked (e.g. small or transcriptionally intermediate clusters) --
exactly the clusters that also tend to show lower agreement with the
SEA-AD reference in Q2, reinforcing that these are the genuinely harder
cases rather than an artifact of one particular annotation method.